# Data download

In [1]:
!wget -O wikitext-filtered-full.zip "https://www.dropbox.com/scl/fi/ibd4cmixckghx6hhb361c/wikitext-filtered-full.zip?rlkey=q71cebf0k5fvvwhmcntoswzhq&dl=1"
!wget -O wikitext-filtered-10k.zip "https://www.dropbox.com/scl/fi/ek174r3sg7qjx0aa9atop/wikitext-filtered-10k.zip?rlkey=zy6jqxv6qsc16lr9qm3ki9uhf&dl=1"

--2026-10-10 11:09:13--  https://www.dropbox.com/scl/fi/ibd4cmixckghx6hhb361c/wikitext-filtered-full.zip?rlkey=q71cebf0k5fvvwhmcntoswzhq&dl=1
Resolving www.dropbox.com (www.dropbox.com)... 162.125.5.18, 2620:100:601c:18::a27d:612
Connecting to www.dropbox.com (www.dropbox.com)|162.125.5.18|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://uc4880c57f42ed8ef1c37ba264e1.dl-eu.dropboxusercontent.com/cd/0/inline/DJsOkyS3V5FoIdk3IzWIdc6o_ESJC7u0BNhRqyZx76xDBAAHCmNTWsd-f0h0nFnWVA0OBeQ-nazLARMeP5zeh9zeI2mORwbcdoKzH3xreMGjxeJSu87w4T6wWJn-FpxbaURf6dgmhZZPJcK9Uv-J0Mf7/file?dl=1# [following]
--2026-10-10 11:09:14--  https://uc4880c57f42ed8ef1c37ba264e1.dl-eu.dropboxusercontent.com/cd/0/inline/DJsOkyS3V5FoIdk3IzWIdc6o_ESJC7u0BNhRqyZx76xDBAAHCmNTWsd-f0h0nFnWVA0OBeQ-nazLARMeP5zeh9zeI2mORwbcdoKzH3xreMGjxeJSu87w4T6wWJn-FpxbaURf6dgmhZZPJcK9Uv-J0Mf7/file?dl=1
Resolving uc4880c57f42ed8ef1c37ba264e1.dl-eu.dropboxusercontent.com (uc4880c57f42ed8ef1c37ba264e1.dl-eu.dropbo

In [2]:
!unzip wikitext-filtered-full.zip
!unzip wikitext-filtered-10k.zip

Archive:  wikitext-filtered-full.zip
   creating: wikitext-filtered-full/
  inflating: wikitext-filtered-full/dataset_info.json  
  inflating: wikitext-filtered-full/state.json  
  inflating: wikitext-filtered-full/data-00000-of-00001.arrow  
Archive:  wikitext-filtered-10k.zip
   creating: wikitext-filtered-10k/
  inflating: wikitext-filtered-10k/dataset_info.json  
  inflating: wikitext-filtered-10k/state.json  
  inflating: wikitext-filtered-10k/data-00000-of-00001.arrow  


In [3]:
!pip install datasets
import datasets

In [4]:
from datasets import load_dataset, Dataset

def load_dataset():
  wikitext_small = "wikitext-filtered-10k"
  wikitext_large = "wikitext-filtered-full"

  dataset_small = Dataset.load_from_disk(wikitext_small)
  dataset_large = Dataset.load_from_disk(wikitext_large)
  print("wikitext_small: {} docs, wikitext_large: {} docs".format(len(dataset_small), len(dataset_large)))
  return dataset_small, dataset_large

wikitext_small, wikitext_large = load_dataset()

wikitext_small: 10000 docs, wikitext_large: 859955 docs


# Callback: EpochLossLogger

In [5]:
!pip install -q gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 62.2 MB/s eta 0:00:00


In [6]:
from gensim.models.callbacks import CallbackAny2Vec

class EpochLossLogger(CallbackAny2Vec):
    """Callback to log loss after each epoch."""

    def __init__(self):
        self.epoch = 0
        self.prev_loss = 0.0

    def on_epoch_end(self, model):
        loss = model.get_latest_training_loss()
        epoch_loss = loss - self.prev_loss

        print(
            f"Epoch {self.epoch + 1:02d} | "
            f"epoch loss = {epoch_loss:.2f} | "
            f"cumulative loss = {loss:.2f}"
        )

        self.prev_loss = loss
        self.epoch += 1

loss_logger = EpochLossLogger()

The callback is used as follows:


```
model = Word2Vec(
    ...
    compute_loss = True,
    callbacks=[loss_logger],
    ...
    )
```



In [7]:
# step 2

from gensim.models import Word2Vec

small_model = Word2Vec(sentences=wikitext_small, vector_size=50, window=5, min_count=5, workers=8, epochs=5)
small_model.save("wikitext_small.model")

large_model = Word2Vec(sentences=wikitext_large, vector_size=50, window=5, min_count=5, workers=8, epochs=5)
large_model.save("wikitext_large.model")